In [2]:
import json
import sys
from rich import print as rp
from pathlib import Path
import re
from datetime import datetime
from collections import Counter

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

In [3]:
folder_parsed = Path(project_root / "data/parsed")
books_db_file = Path(project_root / "data_reload/db_exports/books-2026-08-07.json")
admin_db_file = Path(project_root / "data_reload/db_exports/book_admin-2026-08-07.json")
topics_db_file = Path(project_root / "data_reload/db_exports/topics-2026-08-07.json")

folder_raw = Path(project_root / "data_reload/reload_backup/raw")
folder_fremdspr = Path(project_root / "data/raw/batched/fremdsprachige")
folder_batched = Path(project_root / "data/raw/batched")
fremdspr_raw_file = Path(project_root / "data_reload/reload_backup/raw/fremdsprachige.json")

missing_log_file = Path(project_root / "data_reload/logs/find_missing_books.json")

In [4]:
with open(topics_db_file, "r") as f:
    topics = json.load(f)

topic_id_lookup = {topic["topic_id"]: topic["topic_normalised"] for topic in topics}
# rp(topic_id_lookup)
topic_norm_lookup = {topic["topic_normalised"]: topic["topic_id"] for topic in topics}


cid_db_set = set()
with open(books_db_file, "r") as f:
   books_db = json.load(f)

db_book_count = 0

for book in books_db:
    db_book_count += 1
    cid_db_set.add(book["composite_id"])

rp(db_book_count)
# with open("cid_db_set.json", "w") as f:
#     json.dump(list(cid_db_set), f, ensure_ascii=False, indent=2)


10919

In [5]:
file_count = 0
parsed_dict = {}
parsed_id_set = set()
references_dict = {}
references_set = set()

line_count = 0
entry_count = 0
reference_count = 0

missing_log = {
    "checking_basics": {},
    "totals": {}
}

for file in folder_parsed.iterdir():
    file_count += 1
    file_reference_count = 0
    file_entry_count = 0
    # people_count = 0
    # author_count = 0

    if not file.exists():
        raise FileNotFoundError(f"{file} doesn't exist!")


    parts = file.stem.split("_")
    topic = parts[1]


    with open(file, "r") as f:
       entries = json.load(f)

    for entry in entries:
        line_count += 1

        composite_id = entry["custom_id"]

        parsed = entry["parsed_entry"]
        problematic_multi_volume = parsed["administrative"]["problematic_multi_volume"]
        is_reference = parsed["administrative"]["is_reference"]

        if is_reference == True:
            file_reference_count += 1
            reference_count  += 1
            references_dict[composite_id] = entry
            references_set.add(composite_id)
        else:
            file_entry_count +=1
            entry_count += 1
            parsed_id_set.add(composite_id)

    if topic not in missing_log["checking_basics"]:
        missing_log["checking_basics"][topic] = {"entries": 0, "references": 0}
    missing_log["checking_basics"][topic]["entries"] += file_entry_count
    missing_log["checking_basics"][topic]["references"]  += file_reference_count


missing_log["totals"] = {
    "files processed": file_count,
    "lines processed": line_count,
    "entries": entry_count,
    "parsed ids in set": len(parsed_id_set),
    "references": reference_count,
    "references in set": len(references_set)
}


# with open("references_set.json", "w") as f:
#     json.dump(list(references_set), f, ensure_ascii=False, indent=2)

# rp(missing_log)


In [6]:
cid_not_in_db = parsed_id_set - cid_db_set
# rp(len(cid_not_in_db))
cid_in_both = parsed_id_set & cid_db_set
# rp(f"cid in both: {len(cid_in_both)}")


In [7]:
raw_info = {}
erstausgaben1 = 0
erstausgaben2 = 0
erstausgaben3 = 0

for file in folder_raw.iterdir():
    with open(file, "r") as f:
        raw = json.load(f)
    topic_raw = file.stem
    count = len(raw)
    if topic_raw == "erstausgaben1":
        erstausgaben1 = count
    elif topic_raw == "erstausgaben2":
        erstausgaben2 = count
    elif topic_raw == "erstausgaben3":
        erstausgaben3 = count
    else:
        raw_info[topic_raw] = count

ea_count = erstausgaben1 + erstausgaben2 + erstausgaben3
raw_info["erstausgaben"] = ea_count

inconsistencies = {}

file_nr = 0
topic_entries = Counter()
topic_files = Counter()
check_counts = Counter()
raw_lookup_complete = {}
cids_raw_set = set()

# for file in folder_fremdspr.iterdir():
for file in folder_batched.rglob("*.json"):

    with open(file, "r") as f:
        entries = json.load(f)
    file_nr += 1
    entry_count = 0
    filename = file.stem
    topic = filename.split("_")[0]
    topic_files[topic] += 1
    topic_entries[topic] += len(entries)

    expected_total = raw_info[topic]

    # rp(expected_total)
    batches = filename.split("_")[1]
    current = int(batches.split("-")[0])
    total_b = int(batches.split("-")[1])
    # rp(batches, current, total)

    for entry in entries:
        cid = entry["composite_id"]
        raw_lookup_complete[cid] = entry

rp(len(raw_lookup_complete))
with open("raw_lookup.json", "w") as f:
    json.dump(raw_lookup_complete, f, ensure_ascii=False, indent=2)


## ========= THIS PART CHECKED FOR INCONSISTENCIES ====
## ========= THERE WERE NONE ====
    # for i, entry in enumerate(entries):
    #     cid = entry["composite_id"]
        # if not cid:
        #     if "no_cid" not in inconsistencies:
        #         inconsistencies["no_cid"] = {}
        #     inconsistencies["no_cid"][cid] = entry
        #     check_counts["no_cid"] += 1

        # cid_parts = cid.split("_")
        # cid_i = int(cid_parts[-3])

        # calc_i = (current - 1) * 25 + i

        # if calc_i != cid_i:
        #     if topic not in inconsistencies[topic]:
        #         inconsistencies[topic] = {}
        #     inconsistencies[topic][cid] = entry
        #     check_counts["inconsistent"] += 1
        # else:
        #     if topic not in cids_per_topic:
        #         cids_per_topic[topic] = []
        #     cids_per_topic[topic].append(cid)
        #     check_counts[topic] += 1
        #     cids_raw_set.add(cid)


# for topic, expected in raw_info.items():
#     if expected != topic_entries[topic]:
#         rp(f"{topic} expected: {expected}, counter total: {topic_entries[topic]}")

# rp(len(cids_raw_set))
# with open("cids_raw_set.json", "w") as f:
#     json.dump(list(cids_raw_set), f, ensure_ascii=False, indent=2)

# rp(check_counts)


12492